# Web Information Retrieval WS 2026 - PyTerrier Tutorial - Part 2

In this second tutorial, we will explore the LongEval Retrieval Test Collection.


For this, we use [PyTerrier](https://pyterrier.readthedocs.io/en/latest/index.html) and [Pandas](https://pandas.pydata.org/). PyTerrier is a practical retrieval toolkit that makes it quick and easy to create and test retrieval systems. The focus is on experimental evaluations, so the systems are not designed to scale to production level. In return, many practical tools are provided to enable rapid evaluation.

More information about the dataset and the shared task can be found here, for example:
- [LongEval Overview](https://clef-longeval.github.io/2025/data/)

In this first hands-on notebook, we will look at the topics, documents, and qrels of the test collection while learning how to use PyTerrier.

First, we need to download the LongEval dataset. Conveniently, we created an integration for the `ir_datasets` toolkit to make it easier for you to use.

First, we need to install PyTerrier again. To do this, simply run this cell:

In [ ]:
!pip install -q python-terrier ir_datasets_longeval

Before we can use PyTerrier, we need to import and initialize it.

In [ ]:
import pyterrier as pt

First, we load the dataset. To do this, we can use the `load` function in `ir_datasets_longeval`. For this test, we only load the *train* slice of the `2024-11` snapshot. In total, about 2 GB need to be downloaded, so this may take a moment.

In [ ]:
from ir_datasets_longeval import load

In [ ]:
dataset = load("longeval-sci-2026/snapshot-1/train/dctr")

To search the dataset, we first need to index it. PyTerrier provides the ability to index datasets in different formats. The small LongEval dataset is available in TREC format (XML). It is best to inspect the files in the file browser to get an overview of the available information.

To index TREC collections, we use the `TRECCollectionIndexer`. Details and an overview of other indexers can be found in the [PyTerrier documentation](https://pyterrier.readthedocs.io/en/latest/terrier-indexing.html#treccollectionindexer).

In [ ]:
indexer = pt.IterDictIndexer(
    index_path="./index", # path to the index
    overwrite=True, 
    meta={"docno": 100, "text": 20480}  # add the docno and text fields to the index metadata
)

In [ ]:
# you can do some custom document processing here
def document_generator():
    for doc in dataset.docs_iter():
        yield {
            "docno": doc.doc_id,
            "text": doc.default_text()
        }

docs = document_generator()

In [ ]:
indexer.index(docs)

The indexing process takes about 10 minutes. You should now have a folder named `index` containing the completed index. Using the following function, we can load the index from the folder and then search it.

In [ ]:
index = pt.IndexFactory.of("./index")

To verify that everything worked correctly, we display the number of documents in the index. There should be 869,902 documents.

In [ ]:
print(index.getCollectionStatistics().toString())